# DeepReviewer — exemplo no Google Colab

Notebook de referência para rodar o `DeepReviewer` refatorado (backend **transformers**, sem vLLM) em qualquer runtime Colab.

**Runtime recomendado:** `Runtime → Change runtime type → GPU (A100 com high-RAM)`.
Em GPUs menores (T4 16 GB, L4 22 GB), use `load_in_4bit=True` ou `model_size="7B"`.

O que este notebook faz:
1. Inspeciona o hardware e instala as dependências.
2. Clona o repositório `Researcher` e instala em modo editável.
3. Carrega `DeepReviewer-14B` com parâmetros auto-detectados.
4. Executa uma revisão em **Fast Mode** sobre um paper de exemplo.
5. Mostra a revisão estruturada (ratings, strengths, weaknesses, decision).

## 1. Inspecionar hardware

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv || echo 'GPU indisponivel - o modelo sera carregado em CPU (muito lento).'

## 2. Clonar o projeto e instalar dependências

Troque `REPO_URL` pela URL do fork da tese se estiver usando uma versão customizada.

In [ ]:
REPO_URL = "https://github.com/ResearAI/Researcher.git"

%cd /content
![ -d Researcher ] || git clone $REPO_URL
%cd /content/Researcher
!pip install -q -e . 2>&1 | tail -n 5

### (opcional) flash-attention para A100/H100
Instala o kernel mais rápido. Se falhar, o código cai automaticamente em `sdpa`.

In [ ]:
# Descomente se estiver em A100/H100 e quiser o ganho de ~2x
# !pip install -q flash-attn --no-build-isolation

### (opcional) bitsandbytes para GPU pequena
Necessário se for usar `load_in_4bit=True` em GPU com <24 GB.

In [ ]:
# !pip install -q bitsandbytes

## 3. Carregar o DeepReviewer

Os parâmetros abaixo são auto-detectados quando deixados em `None`:
- `device` → `cuda` / `mps` / `cpu`
- `torch_dtype` → `bf16` em Ampere+, `fp16` em T4/V100, `fp32` em CPU
- `attn_implementation` → `sdpa` (fallback robusto)

Em A100 você pode subir `max_input_length` para 32768 e passar `attn_implementation="flash_attention_2"`.

In [ ]:
from revisa import DeepReviewer

dr = DeepReviewer(
    model_size="14B",          # use "7B" para GPUs menores
    # load_in_4bit=True,       # descomente em GPU < 24 GB
    # max_input_length=32768,  # aumente em A100/H100
    # attn_implementation="flash_attention_2",
)

## 4. Carregar um paper de exemplo

Monta o texto do paper a partir de `Tutorial/demo_data.json`. Para testar com um paper seu, substitua `paper_text` pelo conteúdo em markdown ou texto puro.

In [ ]:
import json

with open("Tutorial/demo_data.json") as f:
    demo = json.load(f)

paper = demo[0]

def build_paper_text(p):
    parts = [f"# {p['title']}\n\n## Abstract\n{p['abstract']}\n"]
    for sec in p.get("sections", []):
        heading = sec.get("heading") or sec.get("section_name") or "Section"
        body = sec.get("text") or sec.get("content") or ""
        parts.append(f"\n## {heading}\n{body}\n")
    return "".join(parts)

paper_text = build_paper_text(paper)
print(f"Paper: {paper['title']}")
print(f"Tamanho: {len(paper_text):,} chars\n")
print(paper_text[:800], "...")

## 5. Executar a revisão (Fast Mode)

`Fast Mode` é o mais rápido — 1 chamada ao LLM, sem simulação múltipla.
Ideal para o primeiro smoke test.

In [ ]:
import time

t0 = time.time()
results = dr.evaluate(
    paper_text,
    mode="Fast Mode",
    max_tokens=4096,   # Fast Mode precisa de menos tokens
)
print(f"Duracao: {time.time() - t0:.1f} s")
print(f"Reviews retornadas: {len(results)}")

In [ ]:
review = results[0]
print("=== RAW REVIEW ===\n")
print(review["raw_text"][:3000])

## 6. (opcional) Standard Mode com 4 revisores simulados

Mais lento (~4–8× Fast Mode), mas produz ratings, strengths, weaknesses por revisor + meta-review.
Em A100 80GB, roda em ~3–6 minutos para um paper típico.

In [ ]:
t0 = time.time()
results_std = dr.evaluate(
    paper_text,
    mode="Standard Mode",
    reviewer_num=4,
    max_tokens=8192,
)
print(f"Duracao: {time.time() - t0:.1f} s\n")

std = results_std[0]
for r in std["reviews"]:
    print(f"--- Reviewer {r['reviewer_id']} ---")
    print(f"Rating: {r.get('rating', 'N/A')}")
    print(f"Summary: {(r.get('summary') or '')[:300]}")
    print()

if std.get("meta_review"):
    print("=== META REVIEW ===")
    print(f"Rating: {std['meta_review'].get('rating', 'N/A')}")
    print(f"Decision: {std.get('decision', 'N/A')}")
    print(f"Summary: {(std['meta_review'].get('summary') or '')[:500]}")

## 7. Salvar a revisão em JSON

In [ ]:
from pathlib import Path

out = Path("review_output.json")
out.write_text(json.dumps(results_std[0], ensure_ascii=False, indent=2))
print(f"Salvo em {out.resolve()} ({out.stat().st_size:,} bytes)")

---

### Notas sobre o modo Best Mode

O Best Mode depende do serviço externo **OpenScholar** (`http://127.0.0.1:38015`), que não está disponível no Colab por padrão. Para a tese, a recomendação é trocar essa dependência por uma API de busca (Tavily/Perplexity) ou desabilitar o Best Mode nesta fase.